In [4]:
import os
import finaletoolkit
from tqdm import tqdm
import glob
import numpy as np


In [5]:
def read_tss_create_intervals(path):
    with open(path, 'rt') as tss:
        for line in tqdm(tss):
            intervals = []
            chrom, start, end, _, _, strand = line.strip().split('\t')
            start = int(start)
            end = int(end)
            if strand=="+":
                interval_range = range(start-2000, start+2001)
            elif strand=="-":
                interval_range = reversed(range(end-2000, end+2001))
            else:
                raise ValueError(f"There is an error in the formatting of your TSS data. Expected either '+' or '-' for the strand. Got {strand}")
            for interval_start in interval_range:
                intervals.append([chrom[3:], interval_start, interval_start+1])
            filename = f"./tmp/{chrom}_{start}_{end}.tss.intervals.bed"
            with open(filename, 'w') as output:
                for interval in intervals:
                    output.write(f"{interval[0]}\t{interval[1]}\t{interval[2]}\n")

In [6]:
read_tss_create_intervals("../data/tss.hg19.bed")

17880it [00:58, 305.07it/s]


In [7]:
with open("../data/tss.hg19.bed", 'rt') as ctcf, open('tss.commands.txt', 'w') as command_file:
    for line in tqdm(ctcf):
        chrom, start, end, _, _, strand = line.strip().split('\t')
        filename = f"/jet/home/rbandaru/ravi/finaletoolkit_figures/2B/tmp/{chrom}_{start}_{end}.tss.intervals.bed"
        out_filename = f"/jet/home/rbandaru/ravi/finaletoolkit_figures/2B/tmp/{chrom}_{start}_{end}.tss.intervals.wcov.bed"
        #if not os.path.exists(out_filename):
        command = [
            '/jet/home/rbandaru/ravi/.conda/envs/ravi-ftk/bin/finaletools', 'coverage-intervals', 
            '-o', out_filename, 
            '-q', '30',
            '-w', '1',
            '-i', 'any',
            '/jet/home/rbandaru/ravi/finaletoolkit_figures/data/BH01.hg19.mdups.bam', 
            filename
        ]
        command_str = ' '.join(command)
        command_file.write(command_str + '\n')

17880it [00:00, 635452.14it/s]


In [8]:
from tqdm import tqdm

# Open the input file containing commands and the output shell script
with open('tss.commands.txt', 'r') as command_file, open('submit_tss_jobs.sh', 'w') as sbatch_script:
    # Write a shebang to make the script executable (optional, for direct execution)
    sbatch_script.write("#!/bin/bash\n\n")
    
    # Buffer to store commands in chunks of 10
    command_buffer = []
    
    # Process each line in the command file
    for line in tqdm(command_file):
        command = line.strip()
        command_buffer.append(command)
        
        # When buffer reaches 10 commands, create an sbatch command
        if len(command_buffer) == 10:
            combined_commands = "; ".join(command_buffer)
            sbatch_command = f"sbatch -p RM-shared -t 3:00:00 --wrap='{combined_commands}'\n"
            sbatch_script.write(sbatch_command)
            command_buffer = []  # Clear the buffer

    # Handle any remaining commands (less than 10)
    if command_buffer:
        combined_commands = "; ".join(command_buffer)
        sbatch_command = f"sbatch -p RM-shared -t 3:00:00 --wrap='{combined_commands}'\n"
        sbatch_script.write(sbatch_command)

# Make the script executable (if running from a Python script)
import os
os.chmod('submit_tss_jobs.sh', 0o755)

print("Job submission script 'submit_tss_jobs.sh' created successfully.")


17880it [00:00, 1052505.24it/s]

Job submission script 'submit_tss_jobs.sh' created successfully.


In [9]:
import os
import glob
import numpy as np
from tqdm import tqdm

directory = "/jet/home/rbandaru/ravi/finaletoolkit_figures/2B/tmp/"
pattern = "*.tss.intervals.wcov.bed"
file_paths = glob.glob(os.path.join(directory, pattern))

output_dir = "normalized_arrays"
os.makedirs(output_dir, exist_ok=True)

for file_index, file_path in enumerate(tqdm(file_paths)):
    try:
        # Load the file as a numpy array, skipping lines with errors
        data = np.loadtxt(file_path, usecols=[4])  # Load only the 5th column
    except Exception as e:
        print(f"Error processing file {file_path}: {e}")
        continue

    total_sum = np.sum(data)

    if total_sum > 0:
        normalized_column = data
        
        # Save each normalized array individually
        output_file = os.path.join(output_dir, f"normalized_{file_index}.npy")
        np.save(output_file, normalized_column)

  0%|          | 57/17477 [00:01<10:47, 26.90it/s]/var/tmp/ipykernel_58073/1482631948.py:16: UserWarning: loadtxt: input contained no data: "/jet/home/rbandaru/ravi/finaletoolkit_figures/2B/tmp/chr10_48963180_48963181.tss.intervals.wcov.bed"
  data = np.loadtxt(file_path, usecols=[4])  # Load only the 5th column
  1%|          | 148/17477 [00:04<11:07, 25.97it/s]/var/tmp/ipykernel_58073/1482631948.py:16: UserWarning: loadtxt: input contained no data: "/jet/home/rbandaru/ravi/finaletoolkit_figures/2B/tmp/chr10_96943409_96943410.tss.intervals.wcov.bed"
  data = np.loadtxt(file_path, usecols=[4])  # Load only the 5th column
  1%|▏         | 254/17477 [00:09<14:54, 19.25it/s]/var/tmp/ipykernel_58073/1482631948.py:16: UserWarning: loadtxt: input contained no data: "/jet/home/rbandaru/ravi/finaletoolkit_figures/2B/tmp/chr2_48982879_48982880.tss.intervals.wcov.bed"
  data = np.loadtxt(file_path, usecols=[4])  # Load only the 5th column
  3%|▎         | 537/17477 [00:20<20:04, 14.07it/s]/var/t

In [10]:
import os
import glob
import numpy as np
from tqdm import tqdm

# Directory and pattern setup
directory = "/jet/home/rbandaru/ravi/finaletoolkit_figures/2B/tmp/"
pattern = "*.tss.intervals.wcov.bed"
file_paths = glob.glob(os.path.join(directory, pattern))

output_dir = "normalized_arrays"
os.makedirs(output_dir, exist_ok=True)

# Initialize variables for incremental computation
total_sum = None
count = 0

normalized_files = glob.glob(os.path.join(output_dir, "normalized_*.npy"))

for norm_file in tqdm(normalized_files):
    norm_array = np.load(norm_file)
    if total_sum is None:
        # Initialize the total_sum array with the shape of the first array
        total_sum = np.zeros_like(norm_array, dtype=np.float64)
    
    # Incrementally update the sum and count
    total_sum += norm_array
    count += 1

# Compute the average
if count > 0:
    average_array = total_sum / count
    # Save the results
    np.save('tss.normalizedaverages.npy', average_array)
else:
    print("No files found for processing.")


100%|██████████| 17812/17812 [00:13<00:00, 1319.17it/s]
